In [1]:
!pip install pandas numpy scipy scikit-learn tsfresh pyarrow

  Using cached pandas-3.0.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached scikit_learn-1.9.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached charset_normalizer-3.4.7-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached numpy-2.4.6-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
Using cached pandas-3.0.3-cp312-cp312-win_amd64.whl (9.8 MB)
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
    --------------------------------------- 0.5/36.6 MB 3.4 MB/s eta 0:00:11
   - ---------------------------

In [1]:
# ============================================================
# AMPds2 — Full Feature Engineering Pipeline (LOCAL VERSION)
# Sept-Oct 2012 subset | Synthetic anomaly injection |
# TSFresh + FRESH-pruning (real labels) | Context vector
# Runs locally on Windows — cached to disk, survives restarts.
# ============================================================

import pandas as pd
import numpy as np
import os
from tsfresh.utilities.dataframe_functions import roll_time_series, impute
from tsfresh.feature_extraction import extract_features, EfficientFCParameters, MinimalFCParameters
from tsfresh import select_features


# def main():
# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
DATA_DIR = r'C:\1.Revanth\Projects\research'
DATE_START = '2012-09-01'
DATE_END   = '2012-10-31'
WINDOW_SIZE_MIN = 15
STRIDE_MIN      = WINDOW_SIZE_MIN   # MUST match window size — keeps windows non-overlapping and grid-aligned
APPLIANCE_COLS  = ['FRE', 'HPE', 'DWE', 'CWE', 'WOE', 'B1E']
USE_MINIMAL_FEATURES = False   # False = full EfficientFCParameters run (now that we're local)

N_JOBS = 10   # leaves 12 threads free on a 22-thread Core Ultra 7 for OS/other apps
print(f"CPUs available: {os.cpu_count()} | using n_jobs={N_JOBS}")

CACHE_DIR = os.path.join(DATA_DIR, 'pipeline_cache')
os.makedirs(CACHE_DIR, exist_ok=True)
ROLLED_PATH       = os.path.join(CACHE_DIR, 'df_rolled.parquet')
RAW_FEATURES_PATH = os.path.join(CACHE_DIR, 'raw_features.parquet')

np.random.seed(42)

# ============================================================
# 1. LOAD DATA
# ============================================================
whe     = pd.read_csv(os.path.join(DATA_DIR, 'Electricity_WHE.csv'))
p_all   = pd.read_csv(os.path.join(DATA_DIR, 'Electricity_P.csv'))
weather = pd.read_csv(os.path.join(DATA_DIR, 'Climate_HourlyWeather.csv'))
# normals file removed — not used (wide pivot format, not worth parsing for one feature)

# ============================================================
# 2. TIMESTAMP ALIGNMENT
# ============================================================
whe_ts_col = 'unix_ts' if 'unix_ts' in whe.columns else 'UNIX_TS'
p_ts_col   = 'UNIX_TS' if 'UNIX_TS' in p_all.columns else 'unix_ts'

whe['timestamp']   = pd.to_datetime(whe[whe_ts_col], unit='s', utc=True).dt.tz_convert('America/Vancouver')
p_all['timestamp'] = pd.to_datetime(p_all[p_ts_col], unit='s', utc=True).dt.tz_convert('America/Vancouver')

whe   = whe.drop(columns=[whe_ts_col]).set_index('timestamp').sort_index()
p_all = p_all.drop(columns=[p_ts_col]).set_index('timestamp').sort_index()

weather['timestamp'] = pd.to_datetime(weather['Date/Time'], format='mixed')
weather = weather.set_index('timestamp').sort_index()
weather.index = weather.index.tz_localize('America/Vancouver', ambiguous='NaT', nonexistent='shift_forward')
weather = weather[weather.index.notna()]

weather = weather.drop(columns=[
    'Data Quality', 'Temp Flag', 'Dew Point Temp Flag', 'Rel Hum Flag',
    'Wind Dir Flag', 'Wind Spd Flag', 'Visibility Flag', 'Stn Press Flag',
    'Hmdx', 'Hmdx Flag', 'Wind Chill', 'Wind Chill Flag'
], errors='ignore')

# ============================================================
# 3. FILTER TO 2-MONTH SUBSET
# ============================================================
whe     = whe.loc[DATE_START:DATE_END]
p_all   = p_all.loc[DATE_START:DATE_END]
weather = weather.loc[DATE_START:DATE_END]

print(f"WHE rows: {len(whe)}, P rows: {len(p_all)}, weather rows: {len(weather)}")

# ============================================================
# 4. MERGE WHE (mains) + selected appliances from P.csv
# ============================================================
appliance_cols_present = [c for c in APPLIANCE_COLS if c in p_all.columns]
merged = whe.join(p_all[appliance_cols_present], how='inner')

BEHAVIOR_COLS = ['V', 'I', 'P', 'Q', 'S'] + appliance_cols_present

for col in BEHAVIOR_COLS:
    merged[col] = pd.to_numeric(merged[col], errors='coerce').astype('float64')
merged[BEHAVIOR_COLS] = merged[BEHAVIOR_COLS].ffill().bfill()

# ============================================================
# 5. WEATHER: encode regime + merge into minute-level data
# ============================================================
def simplify_weather(w):
    if pd.isna(w):
        return 'Unknown'
    w = w.lower()
    if 'thunder' in w:
        return 'Thunderstorm'
    if 'snow' in w:
        return 'Snow'
    if 'fog' in w:
        return 'Fog'
    if 'rain' in w or 'drizzle' in w:
        return 'Rain'
    if 'cloud' in w:
        return 'Cloudy'
    if 'clear' in w or 'sunny' in w:
        return 'Clear'
    return 'Other'

weather['weather_regime'] = weather['Weather'].apply(simplify_weather)
print("Weather regime distribution:")
print(weather['weather_regime'].value_counts())

weather_dummies = pd.get_dummies(weather['weather_regime'], prefix='wx')
weather = weather.join(weather_dummies)

weather_numeric_cols = ['Temp (C)', 'Rel Hum (%)', 'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)']
for col in weather_numeric_cols:
    weather[col] = pd.to_numeric(weather[col], errors='coerce')

weather_regime_cols = weather_dummies.columns.tolist()

merged = merged.join(weather[weather_numeric_cols + weather_regime_cols], how='left')
merged[weather_numeric_cols + weather_regime_cols] = merged[weather_numeric_cols + weather_regime_cols].ffill().infer_objects(copy=False)

# ============================================================
# 6. SYNTHETIC ANOMALY INJECTION
# ============================================================
merged['window_id'] = merged.index.floor(f'{WINDOW_SIZE_MIN}min')
anomaly_log = []

def inject_heating_on_warm_day(df, log, n_events=15):
    candidates = df[(df['wx_Clear'] == 1) & (df['Temp (C)'] > 15)].index
    if len(candidates) == 0:
        print("No candidates for heating_on_warm_day — skipped")
        return df
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, 'HPE'] = df.loc[window, 'HPE'].max() * np.random.uniform(3, 5) + 500
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'heating_on_warm_day'})
    return df

def inject_night_load_spike(df, log, n_events=15):
    candidates = df[(df.index.hour >= 1) & (df.index.hour <= 4)].index
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, 'P'] = df.loc[window, 'P'] + np.random.uniform(800, 1500)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'night_load_spike'})
    return df

def inject_stuck_appliance(df, log, appliance='DWE', n_events=10, duration_min=60):
    candidates = df.index[:-duration_min]
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    stuck_value = df[appliance].quantile(0.75)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=duration_min))]
        df.loc[window, appliance] = stuck_value
        for w in pd.date_range(ts, periods=duration_min // WINDOW_SIZE_MIN, freq=f'{WINDOW_SIZE_MIN}min'):
            log.append({'window_id': w.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'stuck_appliance'})
    return df

def inject_weather_mismatched_load(df, log, n_events=15):
    candidates = df[(df['wx_Rain'] == 1) & (abs(df['Temp (C)'] - df['Temp (C)'].mean()) < 2)].index
    if len(candidates) == 0:
        print("No candidates for weather_mismatched_load — skipped")
        return df
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, 'P'] = df.loc[window, 'P'] + np.random.uniform(600, 1200)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'weather_mismatched_load'})
    return df

def inject_weekend_pattern_on_weekday(df, log, n_events=10):
    weekday_candidates = df[df.index.dayofweek < 5].index
    weekend_avg = df[df.index.dayofweek >= 5]['CWE'].mean()
    chosen = np.random.choice(weekday_candidates, size=min(n_events, len(weekday_candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, 'CWE'] = weekend_avg * np.random.uniform(1.5, 2.5)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'weekend_pattern_on_weekday'})
    return df

merged = inject_heating_on_warm_day(merged, anomaly_log)
merged = inject_night_load_spike(merged, anomaly_log)
merged = inject_stuck_appliance(merged, anomaly_log)
merged = inject_weather_mismatched_load(merged, anomaly_log)
merged = inject_weekend_pattern_on_weekday(merged, anomaly_log)

merged = merged.drop(columns=['window_id'])

anomaly_df = pd.DataFrame(anomaly_log).drop_duplicates(subset='window_id')
print("\nInjected anomaly counts by type:")
print(anomaly_df['anomaly_type'].value_counts())
print(f"Total anomalous windows: {len(anomaly_df)}")

anomaly_df.to_csv(os.path.join(CACHE_DIR, 'injected_anomaly_labels.csv'), index=False)

# ============================================================
# 7. BUILD RAW MULTIVARIATE STREAM -> LONG FORMAT FOR TSFRESH
# ============================================================
long_df = merged.reset_index().melt(
    id_vars=['timestamp'],
    value_vars=BEHAVIOR_COLS,
    var_name='sensor', value_name='value'
)
long_df['entity_id'] = 'house'   # single constant entity — required so rolling windows contain ALL sensors together

# ============================================================
# 8. SLIDING WINDOW — cached locally
# ============================================================
if os.path.exists(ROLLED_PATH):
    print("Found cached df_rolled — loading instead of recomputing")
    df_rolled = pd.read_parquet(ROLLED_PATH)
else:
    print("No cache found — running roll_time_series (slow step #1)")
    df_rolled = roll_time_series(
        long_df,
        column_id='entity_id',      # was 'sensor' — fixed: rolls the whole house together, not per-sensor
        column_sort='timestamp',
        column_kind='sensor',       # sensor is now correctly treated as the multivariate "kind" dimension
        max_timeshift=WINDOW_SIZE_MIN - 1,
        min_timeshift=WINDOW_SIZE_MIN - 1,
        rolling_direction=STRIDE_MIN
    )
    df_rolled['id'] = df_rolled['id'].astype(str)
    df_rolled.to_parquet(ROLLED_PATH)
    print(f"Saved to {ROLLED_PATH}")

print(f"Total windows to extract: {df_rolled['id'].nunique()}")

# ============================================================
# 9. TSFRESH FEATURE EXTRACTION — cached locally
# ============================================================
if os.path.exists(RAW_FEATURES_PATH):
    print("Found cached raw_features — loading instead of re-extracting")
    raw_features = pd.read_parquet(RAW_FEATURES_PATH)
else:
    print("No cache found — running extract_features (slow step #2, this is the big one)")
    fc_params = MinimalFCParameters() if USE_MINIMAL_FEATURES else EfficientFCParameters()
    raw_features = extract_features(
        df_rolled,
        column_id='id', column_sort='timestamp',
        column_kind='sensor', column_value='value',
        default_fc_parameters=fc_params,
        n_jobs=N_JOBS,
        chunksize=500
    )
    raw_features = impute(raw_features)

    # recover a clean window_id (Timestamp) from the (entity_id, timestamp) tuple-string index
    def parse_window_id(idx_str):
        ts_str = idx_str.split("Timestamp('")[1].split("'")[0]
        return pd.Timestamp(ts_str).floor(f'{WINDOW_SIZE_MIN}min')

    raw_features.index = pd.Index([parse_window_id(i) for i in raw_features.index], name='window_id')
    raw_features = raw_features[~raw_features.index.duplicated(keep='first')]

    raw_features.to_parquet(RAW_FEATURES_PATH)
    print(f"Saved to {RAW_FEATURES_PATH}")

print(raw_features.shape)
print(raw_features.index[:5])

# ============================================================
# 10. BUILD REAL LABEL (from injected anomalies)
# ============================================================
true_target = pd.Series(0, index=raw_features.index)
true_target.loc[true_target.index.isin(anomaly_df['window_id'])] = 1
true_target = true_target.astype(int)

print("Label distribution:")
print(true_target.value_counts())

label_lookup = anomaly_df.set_index('window_id')['anomaly_type']

# ============================================================
# 11. FRESH PRUNING -> BEHAVIOR VECTOR
# ============================================================
behavior_vector = select_features(raw_features, true_target)
print(f"\nFRESH-pruned behavior features: {behavior_vector.shape[1]} (from {raw_features.shape[1]})")

# ============================================================
# 12. CONTEXT VECTOR (cyclic time + weather regime — no historical normals)
# ============================================================
agg_dict = {col: 'mean' for col in weather_numeric_cols}
agg_dict.update({col: 'max' for col in weather_regime_cols})

context = merged.resample(f'{WINDOW_SIZE_MIN}min').agg(agg_dict)

context['hour']      = context.index.hour
context['dayofweek'] = context.index.dayofweek

context['hour_sin']   = np.sin(2 * np.pi * context['hour'] / 24)
context['hour_cos']   = np.cos(2 * np.pi * context['hour'] / 24)
context['dow_sin']    = np.sin(2 * np.pi * context['dayofweek'] / 7)
context['dow_cos']    = np.cos(2 * np.pi * context['dayofweek'] / 7)
context['is_weekend'] = context['dayofweek'].isin([5, 6]).astype(int)

context = context.drop(columns=['hour', 'dayofweek'])

# ============================================================
# 13. ALIGN + CONCATENATE BEHAVIOR + CONTEXT + LABELS
# ============================================================
combined = behavior_vector.join(context, how='inner')
combined = combined.dropna()

combined['is_anomaly'] = combined.index.isin(anomaly_df['window_id']).astype(int)
combined['anomaly_type'] = combined.index.map(label_lookup).fillna('normal')

print(f"\nFinal combined feature matrix: {combined.shape[0]} windows x {combined.shape[1]} columns")
print(combined['is_anomaly'].value_counts())

# ============================================================
# 14. SAVE FINAL OUTPUT
# ============================================================
output_path = os.path.join(CACHE_DIR, 'ampds_behavior_context_labeled_features.csv')
combined.to_csv(output_path)
print(f"Saved: {output_path}")

# if __name__ == '__main__':
#     main()

CPUs available: 22 | using n_jobs=10
WHE rows: 87840, P rows: 87840, weather rows: 1464
Weather regime distribution:
weather_regime
Clear     737
Cloudy    469
Rain      163
Fog        95
Name: count, dtype: int64


C:\Users\revan\AppData\Local\Temp\ipykernel_34860\2745692548.py:125: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  merged[weather_numeric_cols + weather_regime_cols] = merged[weather_numeric_cols + weather_regime_cols].ffill().infer_objects(copy=False)



Injected anomaly counts by type:
anomaly_type
stuck_appliance               40
heating_on_warm_day           15
night_load_spike              15
weather_mismatched_load       12
weekend_pattern_on_weekday    10
Name: count, dtype: int64
Total anomalous windows: 92
No cache found — running roll_time_series (slow step #1)


Rolling: 100%|██████████| 55/55 [01:13<00:00,  1.34s/it]


Saved to C:\1.Revanth\Projects\research\pipeline_cache\df_rolled.parquet
Total windows to extract: 5856
No cache found — running extract_features (slow step #2, this is the big one)


Feature Extraction: 100%|██████████| 129/129 [08:00<00:00,  3.73s/it]
c:\1.Revanth\Projects\research\.venv\Lib\site-packages\tsfresh\utilities\dataframe_functions.py:198: RuntimeWarning: The columns <ArrowStringArray>
[                             'B1E__partial_autocorrelation__lag_7',
                              'B1E__partial_autocorrelation__lag_8',
                              'B1E__partial_autocorrelation__lag_9',
                                 'B1E__spkt_welch_density__coeff_8',
                               'B1E__ar_coefficient__coeff_0__k_10',
                               'B1E__ar_coefficient__coeff_1__k_10',
                               'B1E__ar_coefficient__coeff_2__k_10',
                               'B1E__ar_coefficient__coeff_3__k_10',
                               'B1E__ar_coefficient__coeff_4__k_10',
                               'B1E__ar_coefficient__coeff_5__k_10',
 ...
   'WOE__agg_linear_trend__attr_"slope"__chunk_len_50__f_agg_"var"',
  'WOE__agg_linear

Saved to C:\1.Revanth\Projects\research\pipeline_cache\raw_features.parquet
(5856, 8547)
DatetimeIndex(['2012-09-01 00:00:00-07:00', '2012-09-01 00:15:00-07:00',
               '2012-09-01 00:30:00-07:00', '2012-09-01 00:45:00-07:00',
               '2012-09-01 01:00:00-07:00'],
              dtype='datetime64[us, UTC-07:00]', name='window_id', freq=None)
Label distribution:
0    5764
1      92
Name: count, dtype: int64

FRESH-pruned behavior features: 336 (from 8547)

Final combined feature matrix: 5856 windows x 352 columns
is_anomaly
0    5764
1      92
Name: count, dtype: int64
Saved: C:\1.Revanth\Projects\research\pipeline_cache\ampds_behavior_context_labeled_features.csv
